# Final Preparation — DATASET_ALEX

Notebook de preparação/limpeza do `DATASET_ALEX.csv` (separador `;`) para treino de modelos.

Notas:
- Usa as colunas do export (`[I] ...`, `[T] ...`, e `Target (...)`).
- Faz parsing do volume para float.
- Filtra targets inválidos (`0` / vazio) e classes demasiado pequenas (para o SMOTE não falhar).
- Não grava ficheiros intermédios no disco (os modelos lêem diretamente do `DATASET_ALEX.csv`).

In [3]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:
# -------- 1) Ler dataset (export com separador ;) --------\n
RAW_PATH = 'DATASET_ALEX.csv'\n
df_raw = pd.read_csv(RAW_PATH, sep=';')\n
print('raw shape:', df_raw.shape)\n
df_raw.head(3)\n

In [ ]:
# -------- 2) Parsing de Volume (m³) para float --------\n
VOLUME_COL = '[I] Volume'\n
\n
def parse_volume_m3(x):\n
    if pd.isna(x):\n
        return np.nan\n
    s = str(x).strip()\n
    # exemplos: '0.04 m³', '1,23 m³', '0.04'\n
    s = s.replace('m³', '').replace('m3', '')\n
    s = s.replace(' ', '')\n
    s = s.replace(',', '.')\n
    # manter apenas números/sinal/ponto/expoente\n
    s = re.sub(r'[^0-9eE+\-\.]', '', s)\n
    return pd.to_numeric(s, errors='coerce')\n
\n
df = df_raw.copy()\n
df['Volume_m3'] = df[VOLUME_COL].apply(parse_volume_m3)\n
print('Volume_m3 NaN %:', df['Volume_m3'].isna().mean() * 100)\n

In [ ]:
# -------- 3) Target (Ss_CODE): limpeza + filtro de classes raras (SMOTE-safe) --------\n
TARGET_COL = 'Target (Ss_CODE)'\n
MIN_SAMPLES_PER_CLASS = 6  # para SMOTE(k_neighbors=5) funcionar em todas as classes\n
\n
y = df[TARGET_COL].astype(str).str.strip()\n
# No DATASET_ALEX, '0' é muito frequente e normalmente representa 'sem código'\n
mask_valid = (y.notna()) & (y != '') & (y != '0')\n
df = df.loc[mask_valid].copy()\n
y = y.loc[mask_valid]\n
\n
vc = y.value_counts()\n
keep_classes = vc[vc >= MIN_SAMPLES_PER_CLASS].index\n
df = df.loc[y.isin(keep_classes)].copy()\n
y = y.loc[y.isin(keep_classes)]\n
\n
print('after filtering shape:', df.shape)\n
print('n_classes:', y.nunique())\n
print('min class count:', int(y.value_counts().min()))\n
y.value_counts().head(15)\n

In [ ]:
# -------- 4) Features: numéricas + one-hot de categóricas com baixa cardinalidade --------\n
ID_COL = '[I] IfcGUID'\n
\n
categorical_cols = [\n
    '[I] Workset',\n
    '[I] Category',\n
    '[I] Level',\n
    '[I] Construction Block',\n
    '[T] Classification Number',\n
    '[T] Category',\n
]\n
categorical_cols = [c for c in categorical_cols if c in df.columns]\n
\n
X_cat = pd.get_dummies(df[categorical_cols].astype('string'), prefix=[c.replace('[','').replace(']','') for c in categorical_cols], dummy_na=True)\n
X_num = df[['Volume_m3']].copy()\n
\n
X = pd.concat([X_num, X_cat], axis=1)\n
X = X.apply(pd.to_numeric, errors='coerce')\n
X = X.fillna(0)\n
print('X shape:', X.shape)\n

In [ ]:
# -------- 5) Label encoding do target (sem gravar ficheiros intermédios) --------
le = LabelEncoder()
y_enc = le.fit_transform(y.astype(str))

id_col = '[I] IfcGUID'
if id_col in df.columns:
    base = df[[id_col]].rename(columns={id_col: 'IfcGUID'}).reset_index(drop=True)
else:
    base = pd.DataFrame(index=range(len(df)))

preview = pd.concat(
    [
        base,
        y.reset_index(drop=True).rename('Target_raw'),
        pd.Series(y_enc, name='Target_encoded'),
        X.reset_index(drop=True),
    ],
    axis=1,
)

label_mapping = pd.DataFrame({'label': le.classes_, 'encoded': np.arange(len(le.classes_))})

print('preview shape:', preview.shape)
print('n_classes:', len(le.classes_))
label_mapping.head(10)